In [1]:
import warnings
warnings.filterwarnings("ignore")

# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning / Model Validation 
from surprise import accuracy
from surprise import CoClustering
from surprise.reader import Reader
from surprise.dataset import Dataset
from surprise.prediction_algorithms import KNNBasic
from surprise.prediction_algorithms.matrix_factorization import SVD
from surprise import CoClustering
from surprise.model_selection import KFold
from surprise.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from surprise.model_selection import GridSearchCV
from sklearn.metrics import ndcg_score

from collections import defaultdict

In [2]:
ratings_data = pd.read_parquet("../data/movielens_ratings_w_tmdbId.parquet")

In [3]:
# Instantiate the reader and the dataset, then split the dataset into training and testing sets
reader = Reader(rating_scale=(0.5, 5.0))
dataset = Dataset.load_from_df(ratings_data[['userId', 'tmdbId', 'rating']], reader)
trainset, testset = train_test_split(dataset, test_size=0.2, random_state=42)

In [4]:
# Function to evaluate the model
def precision_recall_at_k(model, k=30, threshold=1.5):

    user_est_true = defaultdict(list)
    predictions = model.test(testset)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold)) for (est, true_r) in user_ratings[:k])

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 1
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 1

    precision = round(sum(prec for prec in precisions.values()) / len(precisions), 3)
    recall = round(sum(rec for rec in recalls.values()) / len(recalls), 3)

    print(f'RMSE: {accuracy.rmse(predictions, verbose=False)}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    print(f'F1 Score: {2 * (precision * recall) / (precision + recall)}')



In [5]:
# Build baseline model
svd_model = SVD(random_state=42)
svd_model.fit(trainset)
precision_recall_at_k(svd_model)

RMSE: 0.7705958249668379
Precision: 0.968
Recall: 0.874
F1 Score: 0.918601520086862


In [7]:
param_grid = {'n_epochs': [10, 20, 30], 'lr_all': [0.001, 0.005, 0.01],
              'reg_all': [0.2, 0.4, 0.6]}

# Performe 3-fold grid-search cross-validation
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)

# Fitting data
gs.fit(dataset)

# Best RMSE score
print(gs.best_score['rmse'])

# Combination of parameters that gave the best RMSE score
print(gs.best_params['rmse'])

KeyboardInterrupt: 